In [1]:
# needed imports
import os
import gzip
import tarfile
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, utils, datasets
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# load in and format the dataset
class CancerDataset(Dataset):
  def __init__(self, root, download=True, size=512, train=True):
    if download and not os.path.exists(os.path.join(root, 'cancer_data')):
      datasets.utils.download_url('http://liftothers.org/cancer_data.tar.gz', root, 'cancer_data.tar.gz', None)
      self.extract_gzip(os.path.join(root, 'cancer_data.tar.gz'))
      self.extract_tar(os.path.join(root, 'cancer_data.tar'))

    postfix = 'train' if train else 'test'
    root = os.path.join(root, 'cancer_data', 'cancer_data')
    self.dataset_folder = datasets.ImageFolder(os.path.join(root, 'inputs_' + postfix) ,transform = transforms.Compose([transforms.Resize(size),transforms.ToTensor()]))
    self.label_folder = datasets.ImageFolder(os.path.join(root, 'outputs_' + postfix) ,transform = transforms.Compose([transforms.Resize(size),transforms.ToTensor()]))

  @staticmethod
  def extract_gzip(gzip_path, remove_finished=False):
    print('Extracting {}'.format(gzip_path))
    with open(gzip_path.replace('.gz', ''), 'wb') as out_f, gzip.GzipFile(gzip_path) as zip_f:
      out_f.write(zip_f.read())
    if remove_finished:
      os.unlink(gzip_path)

  @staticmethod
  def extract_tar(tar_path):
    print('Untarring {}'.format(tar_path))
    z = tarfile.TarFile(tar_path)
    z.extractall(tar_path.replace('.tar', ''))

  def __getitem__(self,index):
    img = self.dataset_folder[index]
    label = self.label_folder[index]
    return img[0],label[0][0]

  def __len__(self):
    return len(self.dataset_folder)

train_dataset = CancerDataset("/tmp/Datasets/cancer", train=True)
val_dataset = CancerDataset("/tmp/Datasets/cancer", train=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=True)

# @torch.no_grad()
# def cancer_detection_accuracy(y_hat, y):
#     y_hat = y_hat.to(device)
#     y = y.to(device)
#     y_hat = y_hat.argmax(1)
#     pred = (y == y_hat).float()
#     return torch.mean(pred).item()

# @torch.no_grad()
# def validation(net, val_loader):

#     average_loss_lst = []
#     accuracy_lst = []

#     net = net.to(device)

#     for x,y in val_loader:

#       x = x.to(device)
#       y = y.to(device)

#       y_hat = net(x)

#       accuracy_lst.append(cancer_detection_accuracy(y_hat, y))
#       y = y.long()
#       average_loss_lst.append(F.cross_entropy(y_hat,y).item())

#     average_loss = np.mean(average_loss_lst)
#     accuracy = np.mean(accuracy_lst)

#     return average_loss, accuracy

# @torch.no_grad()
# def get_prediction(net, x):
#     x = x.to(device)
#     net = net.to(device)
#     y_hat = net(x)
#     pred = torch.argmax(y_hat, dim=1)
#     pred_im = pred.squeeze(0)
#     return pred_im.cpu()

100%|██████████| 2.75G/2.75G [01:16<00:00, 35.8MB/s]


Extracting /tmp/Datasets/cancer/cancer_data.tar.gz
Untarring /tmp/Datasets/cancer/cancer_data.tar


/tmp/ipykernel_775/4044109931.py:42: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  z.extractall(tar_path.replace('.tar', ''))


Set up metrics and validation loop.

In [2]:
@torch.no_grad()
def pixel_accuracy(y_hat, y):
    preds = y_hat.argmax(1)
    return (preds == y).float().mean().item()

@torch.no_grad()
def dice_score(y_hat, y, eps=1e-8):
    preds = y_hat.argmax(1)

    preds = preds.float()
    y = y.float()

    intersection = (preds * y).sum(dim=(1,2))
    union = preds.sum(dim=(1,2)) + y.sum(dim=(1,2))

    dice = (2 * intersection + eps) / (union + eps)
    return dice.mean().item()

@torch.no_grad()
def iou_score(y_hat, y, eps=1e-8):
    preds = y_hat.argmax(1)

    preds = preds.float()
    y = y.float()

    intersection = (preds * y).sum(dim=(1,2))
    union = ((preds + y) > 0).float().sum(dim=(1,2))

    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()

@torch.no_grad()
def validation(net, val_loader):
    net = net.to(device)
    net.eval()

    loss_sum = 0
    acc_sum = 0
    dice_sum = 0
    iou_sum = 0
    n_batches = 0

    for x, y in val_loader:
        x = x.to(device, non_blocking=True)
        y = (y.to(device, non_blocking=True) > 0.5).long()

        y_hat = net(x)

        loss_sum += F.cross_entropy(y_hat, y).item()
        acc_sum += pixel_accuracy(y_hat, y)
        dice_sum += dice_score(y_hat, y)
        iou_sum += iou_score(y_hat, y)

        n_batches += 1

    return [loss_sum / n_batches, acc_sum / n_batches, dice_sum / n_batches, iou_sum / n_batches]

Run validation.

In [ ]:
# model_map = {
#     "base_unet.pth": default_unet,
#     "bottom_and_middle_skip_unet.pth": bottom_and_middle_skip_unet,
#     "five_blocks_unet.pth": five_blocks_unet,

# }

In [3]:
columns = ["model", "loss", "pixel_acc", "dice", "iou"]

results = []

# for path, ModelClass in model_map.items():
#     net = ModelClass().to(device)
#     net.load_state_dict(torch.load(path, map_location=device))

#     loss, acc, dice, iou = validation(net, val_loader)

#     results.append([path, loss, acc, dice, iou])

Run validation for each architecture

In [1]:
# bottom_and_middle_skip_unet.ipynb
# default_unet.ipynb
# five_blocks_unet.ipynb
# four_blocks_unet.ipynb
# no_skip_unet.ipynb
# one_block_more_capacity_unet.ipynb
# one_block_unet.ipynb
# single_skip_bottom_unet.ipynb
# single_skip_first_unet.ipynb
# single_skip_middle_unet.ipynb
# two_blocks_unet.ipynb


NameError: name 'five_blocks_unet' is not defined

In [4]:
# bottom_and_middle_skip_unet.ipynb
path = "bottom_and_middle_skip_unet.pth"

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(64, 64)

        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        x3 = self.down3(self.pool2(x2))

        x4 = self.bottleneck(self.pool3(x3))

        x = self.up3(x4)
        x = torch.cat([x, x3], dim=1)
        x = self.conv3(x)

        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv2(x)

        x = self.up1(x)
        x = self.conv1(x)

        return self.out_conv(x)

net = UNet().to(device)
net.load_state_dict(torch.load(path, map_location=device))

loss, acc, dice, iou = validation(net, val_loader)

results.append([path, loss, acc, dice, iou])

In [5]:
print(results)

[['bottom_and_middle_skip_unet.pth', 0.2875139588828791, 0.9167450056834654, 0.5041024977228203, 0.4569931218985998]]


In [6]:
# default_unet.ipynb
path = "base_unet.pth"

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        x3 = self.down3(self.pool2(x2))

        x4 = self.bottleneck(self.pool3(x3))

        x = self.up3(x4)
        x = torch.cat([x, x3], dim=1)
        x = self.conv3(x)

        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv2(x)

        x = self.up1(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv1(x)

        return self.out_conv(x)

net = UNet().to(device)
net.load_state_dict(torch.load(path, map_location=device))

loss, acc, dice, iou = validation(net, val_loader)

results.append([path, loss, acc, dice, iou])

In [7]:
print(results)

[['bottom_and_middle_skip_unet.pth', 0.2875139588828791, 0.9167450056834654, 0.5041024977228203, 0.4569931218985998], ['base_unet.pth', 0.23885329107923264, 0.9390265155922283, 0.7533337975090201, 0.7043506902727213]]


In [ ]:
# five_blocks_unet.ipynb
path = "five_blocks_unet.pth"

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.down4 = DoubleConv(128, 256)
        self.pool4 = nn.MaxPool2d(2)

        self.down5 = DoubleConv(256, 512)
        self.pool5 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(512, 1024)

        self.up5 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv5 = DoubleConv(1024, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(64, 32)

        self.out_conv = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        x3 = self.down3(self.pool2(x2))
        x4 = self.down4(self.pool3(x3))
        x5 = self.down5(self.pool4(x4))

        x6 = self.bottleneck(self.pool5(x5))

        x = self.up5(x6)
        x = torch.cat([x, x5], dim=1)
        x = self.conv5(x)

        x = self.up4(x)
        x = torch.cat([x, x4], dim=1)
        x = self.conv4(x)

        x = self.up3(x)
        x = torch.cat([x, x3], dim=1)
        x = self.conv3(x)

        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv2(x)

        x = self.up1(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv1(x)

        return self.out_conv(x)

net = UNet().to(device)
net.load_state_dict(torch.load(path, map_location=device))

loss, acc, dice, iou = validation(net, val_loader)

results.append([path, loss, acc, dice, iou])

In [8]:
# four_blocks_unet.ipynb
path = "four_blocks_unet.pth"

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.down4 = DoubleConv(128, 256)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(64, 32)

        self.out_conv = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        x3 = self.down3(self.pool2(x2))
        x4 = self.down4(self.pool3(x3))

        x5 = self.bottleneck(self.pool4(x4))

        x = self.up4(x5)
        x = torch.cat([x, x4], dim=1)
        x = self.conv4(x)

        x = self.up3(x)
        x = torch.cat([x, x3], dim=1)
        x = self.conv3(x)

        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv2(x)

        x = self.up1(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv1(x)

        return self.out_conv(x)

net = UNet().to(device)
net.load_state_dict(torch.load(path, map_location=device))

loss, acc, dice, iou = validation(net, val_loader)

results.append([path, loss, acc, dice, iou])

In [9]:
print(results)

[['bottom_and_middle_skip_unet.pth', 0.2875139588828791, 0.9167450056834654, 0.5041024977228203, 0.4569931218985998], ['base_unet.pth', 0.23885329107923264, 0.9390265155922283, 0.7533337975090201, 0.7043506902727213], ['four_blocks_unet.pth', 0.4416993214611218, 0.9160786040804603, 0.6704288931055502, 0.6349020312455568]]


In [10]:
# no_skip_unet.ipynb
path = "no_skip_unet.pth"

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(128, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(64, 64)

        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x = self.down1(x)
        x = self.pool1(x)

        x = self.down2(x)
        x = self.pool2(x)

        x = self.down3(x)
        x = self.pool3(x)

        x = self.bottleneck(x)

        x = self.up3(x)
        x = self.conv3(x)

        x = self.up2(x)
        x = self.conv2(x)

        x = self.up1(x)
        x = self.conv1(x)

        return self.out_conv(x)

net = UNet().to(device)
net.load_state_dict(torch.load(path, map_location=device))

loss, acc, dice, iou = validation(net, val_loader)

results.append([path, loss, acc, dice, iou])

In [11]:
print(results)

[['bottom_and_middle_skip_unet.pth', 0.2875139588828791, 0.9167450056834654, 0.5041024977228203, 0.4569931218985998], ['base_unet.pth', 0.23885329107923264, 0.9390265155922283, 0.7533337975090201, 0.7043506902727213], ['four_blocks_unet.pth', 0.4416993214611218, 0.9160786040804603, 0.6704288931055502, 0.6349020312455568], ['no_skip_unet.pth', 0.36593227901242, 0.9006485072049227, 0.5852272727273529, 0.5852272727273529]]


In [12]:
# one_block_more_capacity_unet.ipynb
path = "one_block_more_capacity_unet.pth"

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 24)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(24, 48)

        self.up1 = nn.ConvTranspose2d(48, 24, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(48, 24)  # concat (24 + 24)

        self.out_conv = nn.Conv2d(24, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.down1(x)

        x2 = self.bottleneck(self.pool(x1))

        x = self.up1(x2)
        x = torch.cat([x, x1], dim=1)
        x = self.conv1(x)

        return self.out_conv(x)

net = UNet().to(device)
net.load_state_dict(torch.load(path, map_location=device))

loss, acc, dice, iou = validation(net, val_loader)

results.append([path, loss, acc, dice, iou])

In [13]:
# one_block_unet.ipynb
path = "one_block_unet.pth"

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 16)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(16, 32)

        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(32, 16)

        self.out_conv = nn.Conv2d(16, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.down1(x)

        x2 = self.bottleneck(self.pool(x1))

        x = self.up1(x2)
        x = torch.cat([x, x1], dim=1)
        x = self.conv1(x)

        return self.out_conv(x)

net = UNet().to(device)
net.load_state_dict(torch.load(path, map_location=device))

loss, acc, dice, iou = validation(net, val_loader)

results.append([path, loss, acc, dice, iou])

In [14]:
# single_skip_bottom_unet.ipynb
path = "single_skip_bottom_unet.pth"

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(128, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(64, 64)

        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        x3 = self.down3(self.pool2(x2))

        x4 = self.bottleneck(self.pool3(x3))

        x = self.up3(x4)

        x = torch.cat([x, x3], dim=1)
        x = self.conv3(x)

        x = self.up2(x)
        x = self.conv2(x)

        x = self.up1(x)
        x = self.conv1(x)

        return self.out_conv(x)

net = UNet().to(device)
net.load_state_dict(torch.load(path, map_location=device))

loss, acc, dice, iou = validation(net, val_loader)

results.append([path, loss, acc, dice, iou])

In [15]:
# single_skip_first_unet.ipynb
path = "single_skip_first_unet.pth"

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(128, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.down1(x)
        x = self.pool1(x1)

        x = self.down2(x)
        x = self.pool2(x)

        x = self.down3(x)
        x = self.pool3(x)

        x = self.bottleneck(x)

        x = self.up3(x)
        x = self.conv3(x)

        x = self.up2(x)
        x = self.conv2(x)

        x = self.up1(x)

        x = torch.cat([x, x1], dim=1)

        x = self.conv1(x)

        return self.out_conv(x)

net = UNet().to(device)
net.load_state_dict(torch.load(path, map_location=device))

loss, acc, dice, iou = validation(net, val_loader)

results.append([path, loss, acc, dice, iou])

In [16]:
# single_skip_middle_unet.ipynb
path = "single_skip_middle_unet.pth"

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(64, 64)

        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        x3 = self.down3(self.pool2(x2))

        x = self.bottleneck(self.pool3(x3))

        x = self.up3(x)
        x = self.conv3(x)

        x = self.up2(x)

        x = torch.cat([x, x2], dim=1)
        x = self.conv2(x)

        x = self.up1(x)
        x = self.conv1(x)

        return self.out_conv(x)

net = UNet().to(device)
net.load_state_dict(torch.load(path, map_location=device))

loss, acc, dice, iou = validation(net, val_loader)

results.append([path, loss, acc, dice, iou])

In [17]:
# two_blocks_unet.ipynb
path = "two_blocks_unet.pth"

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 16)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(16, 32)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(32, 64)

        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(64, 32)

        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(32, 16)

        self.out_conv = nn.Conv2d(16, out_channels, kernel_size=1)

    def forward(self, x):

        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))

        x3 = self.bottleneck(self.pool2(x2))

        x = self.up2(x3)
        x = torch.cat([x, x2], dim=1)
        x = self.conv2(x)

        x = self.up1(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv1(x)

        return self.out_conv(x)

net = UNet().to(device)
net.load_state_dict(torch.load(path, map_location=device))

loss, acc, dice, iou = validation(net, val_loader)

results.append([path, loss, acc, dice, iou])

In [18]:
# bottom_to_top_unet.ipynb BTT
path = "bottom_to_top_unet.pth"

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(128, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(320, 64)

        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        x3 = self.down3(self.pool2(x2))

        x = self.bottleneck(self.pool3(x3))

        x = self.up3(x)
        x = self.conv3(x)

        x = self.up2(x)
        x = self.conv2(x)

        x = self.up1(x)

        x3_up = F.interpolate(x3, size=x.shape[2:], mode='bilinear', align_corners=False)

        x = torch.cat([x, x3_up], dim=1)

        x = self.conv1(x)

        return self.out_conv(x)

net = UNet().to(device)
net.load_state_dict(torch.load(path, map_location=device))

loss, acc, dice, iou = validation(net, val_loader)

results.append([path, loss, acc, dice, iou])

In [19]:
# bottom_to_top_unet.ipynb TTB
path = "top_to_bottom_unet.pth"

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(320, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(128, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(64, 64)

        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

        self.proj_x1 = nn.Conv2d(64, 64, kernel_size=1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        x3 = self.down3(self.pool2(x2))

        x = self.bottleneck(self.pool3(x3))

        x = self.up3(x)

        x1_down = F.interpolate(self.proj_x1(x1), size=x.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, x1_down], dim=1)

        x = self.conv3(x)

        x = self.up2(x)
        x = self.conv2(x)

        x = self.up1(x)
        x = self.conv1(x)

        return self.out_conv(x)

net = UNet().to(device)
net.load_state_dict(torch.load(path, map_location=device))

loss, acc, dice, iou = validation(net, val_loader)

results.append([path, loss, acc, dice, iou])

In [20]:
df = pd.DataFrame(results, columns=columns)
print(df)

                               model      loss  pixel_acc      dice       iou
0    bottom_and_middle_skip_unet.pth  0.287514   0.916745  0.504102  0.456993
1                      base_unet.pth  0.238853   0.939027  0.753334  0.704351
2               four_blocks_unet.pth  0.441699   0.916079  0.670429  0.634902
3                   no_skip_unet.pth  0.365932   0.900649  0.585227  0.585227
4   one_block_more_capacity_unet.pth  0.265832   0.908593  0.345738  0.311165
5                 one_block_unet.pth  0.299262   0.912739  0.477348  0.431095
6        single_skip_bottom_unet.pth  0.351387   0.900226  0.583333  0.583333
7         single_skip_first_unet.pth  0.261607   0.932326  0.739253  0.691194
8        single_skip_middle_unet.pth  0.260720   0.939057  0.741731  0.693989
9                two_blocks_unet.pth  0.301551   0.925882  0.598404  0.546470
10            bottom_to_top_unet.pth  0.303615   0.922435  0.678379  0.643820
11            top_to_bottom_unet.pth  0.340449   0.900495  0.583

In [21]:
df.to_csv("results.csv", index=True)

In [22]:
from google.colab import files
files.download("results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>